# Independent reverse-conditional residual fits
$P(\lambda_{\rm spec}\mid\lambda_{\rm RM},z)$, with the no-evolution global log-location fixed per cluster. One log-normal scatter or a three-parameter mean-centered negative-tail mixture in each cell. The forward projection fits are not used. Run the existing fitter with `--cells`, then the summary script with `--cells`, before running this notebook.

## Configuration, parameter tables and convergence

In [6]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2
from IPython.display import display
import emcee
import pygtc

ROOT = Path.cwd().resolve()
if ROOT.name == "richness_relation":
    ROOT = ROOT.parent
assert (ROOT/"richness_relation").is_dir()
sys.path.insert(0,str(ROOT/"richness_relation"))
from prepare_conditional_richness import load_sample
from global_spec_richness_model import residual_logpdf

FITS = ROOT/"catalogs/spec_residual_cells"
OUTPUT = ROOT/"plots/spec_residual_cells"
OUTPUT.mkdir(parents=True,exist_ok=True)
RM_EDGES = [20,30,50,100]
Z_EDGES = [.10,.18,.24,.35]
COLORS = ["crimson","darkorange","royalblue"]
plt.rcParams.update({"font.family":"DejaVu Serif","font.size":12,"axes.labelsize":16,
                     "axes.linewidth":1.1})
comparison=pd.read_csv(FITS/"cell_comparison.csv")
parameters=pd.read_csv(FITS/"cell_parameters.csv")
display(comparison,parameters)
cache={}
for iz in range(3):
    for ir in range(3):
        for model in ("lognormal","mixture"):
            path=FITS/f"z{iz}_rm{ir}"/model
            meta=json.loads((path/"run.json").read_text())
            assert meta["rm_bounds"]==RM_EDGES[ir:ir+2] and meta["z_bounds"]==Z_EDGES[iz:iz+2], \
                "Plot labels must match fitted cell boundaries"
            with np.load(path/"posterior.npz") as f:
                samples=f["samples"].copy()
            diag=json.loads((path/"diagnostics.json").read_text())
            cache[iz,ir,model]=(meta,samples,diag)
meta=cache[0,0,"lognormal"][0]
data,audit=load_sample(meta["data_path"])
assert audit["sample_sha256"]==meta["sample_sha256"]
a,b=meta["fixed_global_mle"]
residual=np.log(data["lambda_spec"])-a-b*np.log(data["lambda_rm"]/40)
print("Fixed global MLE:",a,b)
print("All cell convergence checks pass:",bool(comparison.converged.all()))
print("Histogram chi-square is descriptive; no uncalibrated chi-square p-values are used.")
def style(ax):
    ax.grid(False)
    ax.tick_params(which="both",direction="in",top=True,right=True)
def save(fig,name):
    for ext in ("png","pdf"):
        fig.savefig(OUTPUT/f"{name}.{ext}",dpi=200,bbox_inches="tight")

ImportError: cannot import name 'residual_logpdf' from 'global_spec_richness_model' (/Users/zzbenjamin94/Documents/Codex/2026-05-11/github-plugin-github-openai-curated-can/DESI_c3clusters/richness_relation/global_spec_richness_model.py)

## Residual distributions
Hollow points: histogram densities with approximate 68.27% Poisson errors. Curves and shading: posterior median density and 68% pointwise interval, not data predictive intervals. No normalization to the visible range.

In [ ]:
# Residuals use natural logs. No hidden conversion to dex.
EDGES=np.linspace(-2.0,2.0,23)
GRID=np.linspace(EDGES[0],EDGES[-1],500)
N_DRAWS=300
rng=np.random.default_rng(42)
fig,axes=plt.subplots(3,3,figsize=(12,10),sharex=True)
for iz in range(3):
    for ir in range(3):
        ax=axes[iz,ir]
        meta,_,_=cache[iz,ir,"lognormal"]
        keep=np.isin(data["ID"].astype(str),meta["cluster_ids"])
        values=residual[keep]
        counts,_=np.histogram(values,EDGES)
        low=np.zeros_like(counts,dtype=float)
        nonzero=counts>0
        low[nonzero]=.5*chi2.ppf(.15865,2*counts[nonzero])
        high=.5*chi2.ppf(.84135,2*(counts+1))
        scale=len(values)*np.diff(EDGES)
        ax.errorbar((EDGES[1:]+EDGES[:-1])/2,counts/scale,
                    yerr=np.array([counts-low,high-counts])/scale,
                    fmt="o",ms=3.5,mfc="none",color=COLORS[iz],capsize=0,label="Data")
        for model,color,ls,label in [("lognormal","#2a6298","--","Log-normal"),
                                     ("mixture","0.25","-","Centered mixture")]:
            _,samples,diag=cache[iz,ir,model]
            draws=samples[rng.integers(len(samples),size=N_DRAWS)]
            densities=np.array([np.exp(residual_logpdf(GRID,t,model)) for t in draws])
            lo,mid,hi=np.percentile(densities,[16,50,84],axis=0)
            ax.plot(GRID,mid,color=color,ls=ls,lw=2,label=label)
            ax.fill_between(GRID,lo,hi,color=color,alpha=.13)
        ax.text(.97,.96,rf"${RM_EDGES[ir]}\leq\lambda_{{\rm RM}}<{RM_EDGES[ir+1]}$"
                +"\n"+rf"${Z_EDGES[iz]:.2f}\leq z<{Z_EDGES[iz+1]:.2f}$",
                transform=ax.transAxes,ha="right",va="top",fontsize=11)
        if not all(cache[iz,ir,m][2]["converged"] for m in ("lognormal","mixture")):
            ax.text(.03,.94,"Provisional",transform=ax.transAxes,va="top",fontsize=10)
        ax.set_xlim(EDGES[0],EDGES[-1])
        style(ax)
        if ir==0:ax.set_ylabel("Probability density")
        if iz==2:ax.set_xlabel(r"$\ln\lambda_{\rm spec}-m_{\rm spec}$")
handles,labels=axes[0,0].get_legend_handles_labels()
fig.legend(handles,labels,loc="lower center",ncol=3,frameon=False,bbox_to_anchor=(.5,0))
fig.tight_layout(rect=[0,.045,1,1])
save(fig,"conditional_reverse_residuals")
plt.show()

## Posterior distributions and pyGTC contours

In [ ]:
# One-dimensional scatter posteriors for the log-normal.
fig,axes=plt.subplots(3,3,figsize=(11,8))
for iz in range(3):
    for ir in range(3):
        ax=axes[iz,ir];meta,samples,diag=cache[iz,ir,"lognormal"]
        ax.hist(np.exp(samples[:,0]),bins=50,density=True,histtype="step",color=COLORS[iz])
        ax.set_title(rf"${RM_EDGES[ir]}\leq\lambda_{{\rm RM}}<{RM_EDGES[ir+1]}$, "
                     +rf"${Z_EDGES[iz]:.2f}\leq z<{Z_EDGES[iz+1]:.2f}$",fontsize=10)
        ax.set_xlabel(r"$s_{\rm rev}$")
        style(ax)
fig.tight_layout()
save(fig,"lognormal_cell_posteriors")
plt.show()

# Independent ensembles overlaid, not separate models on incompatible axes.
for iz in range(3):
    for ir in range(3):
        meta,posterior,diag=cache[iz,ir,"mixture"]
        ensembles=[]
        lengths=np.asarray(diag["retained_steps"])*meta["walkers"]
        assert lengths.sum()==len(posterior), "Rerun summaries; posterior metadata differ"
        # Summary preserves ensemble blocks; no live HDF files need to be opened.
        for samples in np.split(posterior,np.cumsum(lengths)[:-1]):
            samples=samples[::max(1,len(samples)//20000)].copy()
            samples[:,0]=np.exp(samples[:,0])
            ensembles.append(samples)
        fig=pygtc.plotGTC(chains=ensembles,
            paramNames=[r"$\sigma_{\rm rev}$",r"$f_{\rm rev}$",r"$\ln\tau_{\rm rev}$"],
            chainLabels=[f"ensemble {i}" for i in range(len(ensembles))],
            figureSize="MNRAS_page",
            customLabelFont={"family":"DejaVu Serif","size":14},
            customLegendFont={"family":"DejaVu Serif","size":10},
            customTickFont={"family":"DejaVu Serif","size":10})
        fig.suptitle(rf"${RM_EDGES[ir]}\leq\lambda_{{\rm RM}}<{RM_EDGES[ir+1]}$, "
                     +rf"${Z_EDGES[iz]:.2f}\leq z<{Z_EDGES[iz+1]:.2f}$",y=1.02,fontsize=12)
        save(fig,f"mixture_corner_z{iz}_rm{ir}")
        print(f"Cell z{iz}_rm{ir}: convergence={diag['converged']}")
        plt.show()